In [1]:
import yaml
import logging
import random
import pandas as pd
import mlflow
import datasets
import torch
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

# logging.basicConfig(filename='sentiment_analysis.log', level=logging.INFO)
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# GLOBAL VARIABLES
config_file = "../configs/run1_sentiment.yaml"
with open(config_file, "r") as f:
    config = yaml.load(f, Loader=yaml.SafeLoader)
dataset_config = config.get("dataset")
model_config = config.get("model")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

rng = random.Random(config.get('seed', 0))

logger.info(f"Configs loaded. Device is set to {device}")

In [9]:
experiment_name = config.get("mlflow", dict).get("experiment", None)
if experiment_name is None:
    raise ValueError(
        "Experiment name is not set in the config.\n"
        "Notebook does not run without tracking"
    )
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='mlflow-artifacts:/13', creation_time=1791120060086, effective_trace_archival_retention=None, experiment_id='13', last_update_time=1791120060086, lifecycle_stage='active', name='llm-bias-variance-2', tags={}, trace_location=None, workspace='default'>

In [10]:
# prep data
dataset=datasets.load_dataset(dataset_config["name"])
split = dataset["train"].train_test_split(train_size=0.7)
train_dataset = split['train']
val_dataset = split['test']
test_dataset = dataset['test']

# Model and Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_config["name"], padding_side='left')
model = AutoModelForCausalLM.from_pretrained(model_config["name"]).to(device)
model.eval()

# Generation Config
generation_config = GenerationConfig(**config.get("generation"))

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 15517.83it/s]


In [27]:
induction_prompt = """You are given a list of inputs and outputs. Your job is to 
infer the nature of the task from the given samples and create instructions for an LLM to correctly complete this task."""
k = 5
selected_samples = rng.choices(train_dataset, k=k)
samples_with_layout = "\n\n".join([f"Input: {sample['review']}\nOutput: {sample['label']}" for sample in selected_samples])
full_prompt = f"## Instruction\n\n{induction_prompt}\n\n## Samples\n\n{samples_with_layout}\n\n### Induction Prompt\n\n" 

# tokenize and generate
tokens = tokenizer([full_prompt], return_tensors="pt").to(model.device)
generated_ids = model.generate(**tokens, generation_config=generation_config)
output = tokenizer.decode(generated_ids, skip_special_tokens=True)


["## Instruction\n\nYou are given a list of inputs and outputs. Your job is to \ninfer the nature of the task from the given samples and create instructions for an LLM to correctly complete this task.\n\n## Samples\n\nInput: You know you're in trouble when the film your watching has numerous alternate titles. Generally it means that they tried and retried to hide the turkey in various markets. Such a turkey is The Brain Machine which has seven different titles.<br /><br />Its about some super secret government project that is suppose to be able to use a computer to read people but instead it drives people to kill each other or themselves, or something like that. Its filled with B level TV actors sitting in paneled room with lawn chairs trying to act a script that makes almost no sense.<br /><br />Its a turkey of the untastey kind. Avoid it.\nOutput: 1\n\nInput: When I was 13 or so I was lucky enough to find this film. It was part of an endless Danish series of really cheesy stuff. This

In [28]:
print(output[0])

## Instruction

You are given a list of inputs and outputs. Your job is to 
infer the nature of the task from the given samples and create instructions for an LLM to correctly complete this task.

## Samples

Input: You know you're in trouble when the film your watching has numerous alternate titles. Generally it means that they tried and retried to hide the turkey in various markets. Such a turkey is The Brain Machine which has seven different titles.<br /><br />Its about some super secret government project that is suppose to be able to use a computer to read people but instead it drives people to kill each other or themselves, or something like that. Its filled with B level TV actors sitting in paneled room with lawn chairs trying to act a script that makes almost no sense.<br /><br />Its a turkey of the untastey kind. Avoid it.
Output: 1

Input: When I was 13 or so I was lucky enough to find this film. It was part of an endless Danish series of really cheesy stuff. This however was

In [ ]:
val_batch_dataset = val_dataset.batch(
    batch_size=1000,
    drop_last_batch=False,
)
with torch.no_grad():
    for batch in val_batch_dataset:
        tokens = tokenizer(
            batch['review'], 
            return_tensors="pt", 
            padding=True
        ).to(model.device)
        output = model.generate(**tokens, generation_config=generation_config)
        break